In the [previous section](/aiml-common/lectures/reinforcement-learning/robot-navigation/ddpg-navigation/index), DDPG learned to reach the goal and then lost that skill again, while its critic predicted returns the robot never obtained. Twin delayed DDPG (TD3) [@Fujimoto2018-uc] keeps the DDPG actor and critic and makes three changes aimed at that overestimation. In this section you add them, train on the same robot, and then use the trained agent to see how the reward weights shape its behavior. The section ends with what changes when the policy moves from this simulation to a real TurtleBot3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

from cycler import cycler
from matplotlib_inline.backend_inline import (
    set_matplotlib_formats,
)

set_matplotlib_formats("svg")
FIG_GRAY = "#808080"
FIG_COLORS = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#cb77a6",
    "#c28600",
    "#1d99e0",
    "#7f7f7f",
]
plt.rcParams.update(
    {
        "figure.facecolor": "none",
        "axes.facecolor": "none",
        "savefig.facecolor": "none",
        "savefig.transparent": True,
        "text.color": FIG_GRAY,
        "axes.labelcolor": FIG_GRAY,
        "axes.edgecolor": FIG_GRAY,
        "xtick.color": FIG_GRAY,
        "ytick.color": FIG_GRAY,
        "grid.color": FIG_GRAY,
        "grid.alpha": 0.25,
        "legend.facecolor": "none",
        "legend.edgecolor": FIG_GRAY,
        "legend.framealpha": 0,
        "axes.prop_cycle": cycler(color=FIG_COLORS),
        "svg.hashsalt": "aiml",
        "svg.fonttype": "path",
    }
)
BLUE, ORANGE, GREEN, PINK, GOLD = FIG_COLORS[:5]

In [ ]:
V_MAX, W_MAX = 0.26, 1.82  # m/s, rad/s (Waffle limits)
RADIUS = 0.2  # m, robot footprint
DT = 0.2  # s, control period
N_BEAMS = 24
R_MIN, R_MAX = 0.12, 3.5  # m, lidar span
ARENA = 4.0  # m, side of the square room
GOAL_TOL = 0.25  # m
T_MAX = 200  # steps per episode (40 s)

# Obstacles: (x, y, radius) in meters
OBSTACLES = np.array(
    [
        [1.2, 1.3, 0.25],
        [2.8, 1.1, 0.20],
        [2.0, 2.4, 0.30],
        [3.0, 3.0, 0.20],
    ]
)


def ray_cast(x, y, angles):
    """Lidar range along each world-frame angle."""
    dx, dy = np.cos(angles), np.sin(angles)
    r = np.full(angles.shape, np.inf)
    with np.errstate(divide="ignore", invalid="ignore"):
        for wall, d, p in (
            (0.0, dx, x),
            (ARENA, dx, x),
            (0.0, dy, y),
            (ARENA, dy, y),
        ):
            t = (wall - p) / d
            r = np.minimum(r, np.where(t > 0, t, np.inf))
    for cx, cy, rho in OBSTACLES:
        ox, oy = x - cx, y - cy
        b = ox * dx + oy * dy
        disc = b * b - (ox * ox + oy * oy - rho * rho)
        t = -b - np.sqrt(np.maximum(disc, 0.0))
        r = np.minimum(
            r, np.where((disc >= 0) & (t > 0), t, np.inf)
        )
    return np.clip(r, R_MIN, R_MAX)


def clearance(x, y):
    """Distance from a point to the nearest surface."""
    walls = min(x, ARENA - x, y, ARENA - y)
    centers = np.hypot(OBSTACLES[:, 0] - x, OBSTACLES[:, 1] - y)
    return min(walls, np.min(centers - OBSTACLES[:, 2]))


class NavEnv:
    """TurtleBot3 Waffle driving to a goal in a 4 m room."""

    def __init__(self, seed=0, noise=0.02, collision_cost=20.0):
        self.collision_cost = collision_cost
        # Starts and goals. Each reset also seeds that episode's noise,
        # so episode k is the same whatever the earlier episodes did.
        self.rng = np.random.default_rng(seed)
        self.noise = noise
        self.beam_angles = np.linspace(
            -np.pi, np.pi, N_BEAMS, endpoint=False
        )

    def _free_point(self, margin):
        while True:
            p = self.rng.uniform(margin, ARENA - margin, size=2)
            if clearance(*p) > margin:
                return p

    def reset(self, start=None, goal=None):
        self.noise_rng = np.random.default_rng(self.rng.integers(2**63))
        if goal is None:
            goal = self._free_point(0.4)
        self.goal = np.array(goal, float)
        if start is None:
            while True:
                p = self._free_point(0.4)
                if np.hypot(*(p - self.goal)) > 1.5:
                    break
            start = (*p, self.rng.uniform(-np.pi, np.pi))
        self.pose = np.array(start, float)
        self.prev = np.zeros(2)
        self.t = 0
        self.dist = self.goal_distance()
        return self.observe()

    def goal_distance(self):
        return float(np.hypot(*(self.goal - self.pose[:2])))

    def goal_bearing(self):
        x, y, phi = self.pose
        g = np.arctan2(self.goal[1] - y, self.goal[0] - x) - phi
        return float(np.arctan2(np.sin(g), np.cos(g)))

    def scan(self):
        x, y, phi = self.pose
        return ray_cast(x, y, phi + self.beam_angles)

    def observe(self):
        """The 28 numbers the agent receives."""
        return np.concatenate(
            [
                self.scan() / R_MAX,
                [
                    min(self.dist, R_MAX) / R_MAX,
                    self.goal_bearing() / np.pi,
                    self.prev[0] / V_MAX,
                    self.prev[1] / W_MAX,
                ],
            ]
        ).astype(np.float32)

    def step(self, v, w):
        v = float(np.clip(v, 0.0, V_MAX))
        w = float(np.clip(w, -W_MAX, W_MAX))
        vn = v + self.noise * self.noise_rng.standard_normal()
        wn = w + self.noise * self.noise_rng.standard_normal()
        x, y, phi = self.pose
        x += vn * np.cos(phi) * DT
        y += vn * np.sin(phi) * DT
        phi = np.arctan2(
            np.sin(phi + wn * DT), np.cos(phi + wn * DT)
        )
        self.pose = np.array([x, y, phi])
        self.prev = np.array([v, w])
        self.t += 1
        d = self.goal_distance()
        progress, self.dist = self.dist - d, d
        r = 2.0 * progress - 0.05 * abs(w) / W_MAX - 0.01
        event, terminated = None, False
        if d < GOAL_TOL:
            r, event, terminated = r + 20.0, "goal", True
        elif clearance(x, y) < RADIUS:
            r -= self.collision_cost
            event, terminated = "collision", True
        truncated = not terminated and self.t >= T_MAX
        if truncated:
            event = "timeout"
        info = {"event": event}
        return self.observe(), r, terminated, truncated, info

In [ ]:
def draw_room(ax, env=None):
    ax.add_patch(
        Rectangle(
            (0, 0), ARENA, ARENA, fill=False, ec=FIG_GRAY, lw=1.5
        )
    )
    for cx, cy, rho in OBSTACLES:
        ax.add_patch(
            Circle((cx, cy), rho, color=FIG_GRAY, alpha=0.5, lw=0)
        )
    if env is not None:
        ax.plot(*env.goal, marker="*", ms=16, color=GOLD, ls="")
    ax.set_xlim(-0.1, ARENA + 0.1)
    ax.set_ylim(-0.1, ARENA + 0.1)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")


def draw_robot(ax, pose, color=BLUE):
    x, y, phi = pose
    ax.add_patch(
        Circle((x, y), RADIUS, fill=False, ec=color, lw=1.5)
    )
    ax.plot(
        [x, x + RADIUS * np.cos(phi)],
        [y, y + RADIUS * np.sin(phi)],
        color=color,
        lw=1.5,
    )


def go_to_goal(env):
    b = env.goal_bearing()
    v = V_MAX * max(0.0, np.cos(b))
    return v, float(np.clip(2.0 * b, -W_MAX, W_MAX))


def rollout(env, policy, start=None, goal=None):
    env.reset(start=start, goal=goal)
    poses, rewards = [env.pose.copy()], []
    while True:
        _, r, term, trunc, info = env.step(*policy(env))
        poses.append(env.pose.copy())
        rewards.append(r)
        if term or trunc:
            return np.array(poses), np.array(rewards), info


def success_rate(policy, n=300, seed=99):
    env = NavEnv(seed=seed)
    events = {"goal": 0, "collision": 0, "timeout": 0}
    for _ in range(n):
        *_, info = rollout(env, policy)
        events[info["event"]] += 1
    return {k: v / n for k, v in events.items()}

In [ ]:
import copy
import torch
import torch.nn as nn

torch.set_num_threads(4)
OBS_DIM, ACT_DIM = N_BEAMS + 4, 2


def mlp(n_in, n_out, out=None):
    layers = [
        nn.Linear(n_in, 256),
        nn.ReLU(),
        nn.Linear(256, 256),
        nn.ReLU(),
        nn.Linear(256, n_out),
    ]
    return nn.Sequential(*layers, *([out] if out else []))


def to_command(a):
    """Rescale an actor output in [-1, 1]^2 to (v, omega)."""
    return (a[0] + 1) / 2 * V_MAX, a[1] * W_MAX


class ReplayBuffer:
    def __init__(self, size=200_000):
        self.o = np.zeros((size, OBS_DIM), np.float32)
        self.a = np.zeros((size, ACT_DIM), np.float32)
        self.r = np.zeros(size, np.float32)
        self.o2 = np.zeros((size, OBS_DIM), np.float32)
        self.term = np.zeros(size, np.float32)
        self.size, self.i, self.n = size, 0, 0

    def add(self, o, a, r, o2, term):
        k = self.i
        self.o[k], self.a[k], self.r[k] = o, a, r
        self.o2[k], self.term[k] = o2, term
        self.i = (k + 1) % self.size
        self.n = min(self.n + 1, self.size)

    def sample(self, batch, rng):
        idx = rng.integers(0, self.n, batch)
        return [
            torch.as_tensor(x[idx])
            for x in (self.o, self.a, self.r, self.o2, self.term)
        ]


def soft_update(net, target, tau):
    with torch.no_grad():
        for p, pt in zip(net.parameters(), target.parameters()):
            pt.mul_(1 - tau).add_(tau * p)


def actor_policy(actor):
    """Wrap a trained actor as a policy for rollout()."""

    def policy(env):
        with torch.no_grad():
            a = actor(torch.as_tensor(env.observe())).numpy()
        return to_command(a)

    return policy


def evaluate(
    actor, critic, n=50, seed=123, gamma=0.99, horizon=1000,
    env_kwargs=None,
):
    """Greedy success rate, the critic's estimate of the
    start-state value, and the return actually obtained.

    The critic bootstraps through the timeout, so the return
    keeps summing rewards past it, until a goal, a collision or
    `horizon` steps (gamma**1000 < 1e-4). Success counts only
    goals reached before the timeout."""
    env = NavEnv(seed=seed, **(env_kwargs or {}))
    goals, q_est, returns = 0, [], []
    for _ in range(n):
        o = env.reset()
        with torch.no_grad():
            s = torch.as_tensor(o)[None]
            q_est.append(
                critic(torch.cat([s, actor(s)], 1)).item()
            )
        G, k, event = 0.0, 0, None
        while k < horizon:
            with torch.no_grad():
                a = actor(torch.as_tensor(o)).numpy()
            o, r, term, trunc, info = env.step(*to_command(a))
            G, k = G + gamma**k * r, k + 1
            if event is None and (term or trunc):
                event = info["event"]
            if term:
                break
        goals += event == "goal"
        returns.append(G)
    return goals / n, np.mean(q_est), np.mean(returns)

## Three changes to DDPG

The overestimation comes from the actor update, which acts as a max over noisy critic estimates. The actor moves its output toward actions where the critic's value is high. Where the critic has an error that happens to be positive, the actor moves there and stays, and the bootstrapped targets $y$ then carry that error back to earlier states. Each change below removes one route for it.

### Clipped double Q-learning

TD3 trains two critics, $Q_{w_1}$ and $Q_{w_2}$, on the same targets, and builds the target from the smaller of their two target networks:

$$
y = r + \gamma\,(1 - \mathbb 1[\text{terminal}]) \min_{i = 1, 2} Q_{w_i'}\big(s', \tilde a'\big).
$$

The same upward bias affects tabular [Q-learning](/aiml-common/lectures/reinforcement-learning/value-based-algorithms/control/q-learning/index), whose target takes a max over noisy table entries. Double Q-learning (van Hasselt, 2010) removed it there with two tables: one picks the best next action, and the other evaluates it. TD3 adapts that idea to two critics and takes the minimum. An error that inflates one critic is unlikely to inflate the other at the same action, so the minimum tends to come out too low. A low bias is safer here: the actor avoids actions that look worse than they are, instead of seeking actions that look better.

### Target policy smoothing

The next action in the target is the target actor's output plus clipped noise,

$$
\tilde a' = \operatorname{clip}\big(\mu_{\theta'}(s') + \operatorname{clip}(\epsilon, -0.5, 0.5),\ -1,\ 1\big), \qquad \epsilon \sim \mathcal N(0, 0.2^2).
$$

The critic then learns the value of a small neighborhood of actions, not of a single point. A narrow spike in $Q$ that the actor could exploit is averaged away. Both this change and the previous one alter only the TD target $y$. The critics still minimize the squared TD error, as in DDPG.

### Delayed policy updates

The critics update on every step, and the actor and all target networks update every second step. The actor then follows a critic whose errors have had time to shrink.

Everything else, from the network sizes to the replay buffer, the exploration noise and the step sizes, is the DDPG setup of the previous section. The training function below marks the three changes.

In [ ]:
def train_td3(
    steps=100_000,
    gamma=0.99,
    tau=0.005,
    batch=256,
    sigma=0.2,
    warmup=5000,
    policy_delay=2,
    target_noise=0.2,
    noise_clip=0.5,
    seed=0,
    env_kwargs=None,
):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    env = NavEnv(seed=seed, **(env_kwargs or {}))
    actor = mlp(OBS_DIM, ACT_DIM, nn.Tanh())
    critic1 = mlp(OBS_DIM + ACT_DIM, 1)
    critic2 = mlp(OBS_DIM + ACT_DIM, 1)  # change 1: twin critics
    actor_t = copy.deepcopy(actor)
    critic1_t = copy.deepcopy(critic1)
    critic2_t = copy.deepcopy(critic2)
    opt_a = torch.optim.Adam(actor.parameters(), lr=1e-4)
    opt_c = torch.optim.Adam(
        list(critic1.parameters()) + list(critic2.parameters()),
        lr=1e-3,
    )
    buf, log, best = ReplayBuffer(), [], (-1.0, None)
    o = env.reset()
    for t in range(steps):
        if t < warmup:
            a = rng.uniform(-1, 1, ACT_DIM)
        else:
            with torch.no_grad():
                a = actor(torch.as_tensor(o)).numpy()
            a = np.clip(
                a + sigma * rng.standard_normal(ACT_DIM), -1, 1
            )
        o2, r, term, trunc, _ = env.step(*to_command(a))
        buf.add(o, a, r, o2, float(term))
        o = env.reset() if term or trunc else o2
        if t < warmup:
            continue
        O, A, R, O2, TERM = buf.sample(batch, rng)
        with torch.no_grad():
            # change 2: target policy smoothing
            eps = (target_noise * torch.randn_like(A)).clamp(
                -noise_clip, noise_clip
            )
            a2 = (actor_t(O2) + eps).clamp(-1, 1)
            x2 = torch.cat([O2, a2], 1)
            # change 1: the smaller of the two target critics
            q_next = torch.min(critic1_t(x2), critic2_t(x2))
            y = R + gamma * (1 - TERM) * q_next.squeeze(1)
        x = torch.cat([O, A], 1)
        loss_c = ((critic1(x).squeeze(1) - y) ** 2).mean() + (
            (critic2(x).squeeze(1) - y) ** 2
        ).mean()
        opt_c.zero_grad()
        loss_c.backward()
        opt_c.step()
        if t % policy_delay == 0:  # change 3: delayed updates
            loss_a = -critic1(torch.cat([O, actor(O)], 1)).mean()
            opt_a.zero_grad()
            loss_a.backward()
            opt_a.step()
            soft_update(actor, actor_t, tau)
            soft_update(critic1, critic1_t, tau)
            soft_update(critic2, critic2_t, tau)
        if (t + 1) % 10_000 == 0:
            rate, q_est, G = evaluate(actor, critic1, env_kwargs=env_kwargs)
            log.append((t + 1, rate, q_est, G))
            if rate > best[0]:
                best = (rate, copy.deepcopy(actor))
    return best[1], np.array(log)


td3_actor, td3_log = train_td3()

In [ ]:
steps_k = td3_log[:, 0] / 1000
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ax.plot(steps_k, td3_log[:, 1], color=GREEN, marker="o")
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("training steps (thousands)")
ax.set_ylabel("greedy success rate")
ax.set_title("TD3: success without exploration noise")
ax2.plot(
    steps_k,
    td3_log[:, 2],
    color=ORANGE,
    marker="o",
    label=r"critic estimate $Q_{w_1}(s_0, \mu_\theta(s_0))$",
)
ax2.plot(
    steps_k,
    td3_log[:, 3],
    color=BLUE,
    marker="o",
    label="return actually obtained",
)
ax2.set_xlabel("training steps (thousands)")
ax2.set_ylabel("discounted return")
ax2.legend(fontsize=8)
ax2.set_title("What the critic believes and what happens")
plt.tight_layout()
plt.show()

*Left: greedy success rate on 50 fixed episodes during TD3 training. Right: the first critic's estimate of the start-state value and the return actually obtained.*

The success rate is zero for the first 30,000 steps, passes 0.9 at 60,000 steps, and stays between 0.94 and 1 from then on. From 30,000 steps on, the critic's estimate stays below the return the robot actually collects. The minimum over two critics errs low, and a low estimate gives the actor no reason to move toward actions that only look good to the critic. On the 300 test episodes:

In [ ]:
td3_policy = actor_policy(td3_actor)
print("go-to-goal:", success_rate(go_to_goal))
print("TD3:       ", success_rate(td3_policy))

In [ ]:
starts = [
    (0.6, 3.4, 0.0),
    (3.6, 1.8, 1.6),
    (0.5, 0.5, 0.0),
    (2.0, 0.5, 1.6),
]
goal = (3.5, 3.5)
fig, (ax, ax2) = plt.subplots(
    1,
    2,
    figsize=(10, 4.6),
    gridspec_kw={"width_ratios": [1, 1.2]},
)
draw_room(ax)
ax.plot(*goal, marker="*", ms=16, color=GOLD, ls="")
env = NavEnv(seed=0)
traces = []
for k, s in enumerate(starts):
    env.reset(start=s, goal=goal)
    poses, cmds = [env.pose.copy()], []
    while True:
        v, w = td3_policy(env)
        _, _, term, trunc, info = env.step(v, w)
        poses.append(env.pose.copy())
        cmds.append((v, w))
        if term or trunc:
            break
    poses = np.array(poses)
    ax.plot(
        poses[:, 0],
        poses[:, 1],
        color=FIG_COLORS[k],
        lw=1.5,
        label=info["event"],
    )
    draw_robot(ax, poses[-1], FIG_COLORS[k])
    traces.append(np.array(cmds))
ax.legend(loc="lower right", fontsize=8)
ax.set_title("TD3 from the four starts")
c = traces[3]
t = np.arange(len(c)) * DT
ax2.plot(t, c[:, 0] / V_MAX, color=BLUE, label=r"$v / v_{\max}$")
ax2.plot(
    t,
    c[:, 1] / W_MAX,
    color=ORANGE,
    label=r"$\omega / \omega_{\max}$",
)
ax2.set_xlabel("time (s)")
ax2.set_ylabel("command, fraction of limit")
ax2.set_ylim(-1.1, 1.1)
ax2.legend(fontsize=8)
ax2.set_title("Commands along the pink trajectory")
plt.tight_layout()
plt.show()

*Left: the TD3 policy from the four starts used in every section. Right: its speed and turn-rate commands along the pink trajectory, each as a fraction of the Waffle's limit.*

Three of the four starts reach the goal. From the bottom left start (green), the robot takes almost the same path as from the pink start and collides with the small obstacle at (3.0, 3.0), short of the goal. The value-iteration policy of the [navigation MDP section](/aiml-common/lectures/reinforcement-learning/robot-navigation/navigation-mdp/index) reached the goal from all four starts, but only for one goal, with the true pose and a model of the dynamics. TD3 needs none of these: it works from the lidar and the goal's relative position, for any goal in the room. From the pink start, the go-to-goal controller collided with the small obstacle at (3.0, 3.0). TD3 drives between the large obstacle and the small one at (2.8, 1.1), then passes below the obstacle at (3.0, 3.0) and reaches the goal from below.

The commands on the right are continuous, but they are not smooth. The speed stays at its limit for the first 8 s and then drops several times, to about a third of its limit near 9 s and 10 s, as the robot turns toward the goal. The turn rate jumps between about $-0.9$ and $+0.8$ of its limit from one step to the next, even on the straight stretch. The reward penalizes turning only a little, and nothing in it penalizes a change of command, so the policy has no reason to be smooth. The simulator applies each command at once, so the jumps do no harm there. On a real robot the motors lag behind the commands, and the robot would shake. Navigation agents for real robots usually add a penalty on the change in $\omega$ to the reward, or pass the commands through a low-pass filter.

## The reward decides the behavior

The page that defined this MDP chose a collision penalty of 20 next to a goal bonus of 20. Here is the same training with the collision penalty raised to 200 and everything else unchanged:

In [ ]:
cautious_actor, cautious_log = train_td3(
    env_kwargs={"collision_cost": 200.0}
)
cautious_policy = actor_policy(cautious_actor)
print("collision cost 200:", success_rate(cautious_policy))

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5.4))
draw_room(ax)
ax.plot(*goal, marker="*", ms=16, color=GOLD, ls="")
env = NavEnv(seed=0)
for k, s in enumerate(starts):
    poses, _, info = rollout(env, cautious_policy, s, goal)
    ax.plot(
        poses[:, 0],
        poses[:, 1],
        color=FIG_COLORS[k],
        lw=1.5,
        label=info["event"],
    )
    draw_robot(ax, poses[-1], FIG_COLORS[k])
ax.legend(loc="lower right", fontsize=8)
ax.set_title("TD3 with collision cost 200")
plt.tight_layout()
plt.show()

*The policy trained with a collision penalty of 200, from the same four starts.*

The agent mostly stops trying. Of the 300 test episodes, 85% end in a timeout, 10% in a collision and 5% at the goal. From the start on the right, below the goal, the robot reaches the goal. From the other three starts it drives part of the way and has not reached the goal when the 40 s run out. This is the best policy it found for the reward it was given. The likely reason is the balance early in training: random actions near obstacles end in collisions that now cost 200, while the goal bonus is still 20 and the robot rarely reaches it. Staying put costs only the time penalty of 0.01 per step, about 2 over a whole episode. Under this reward, doing nothing is a reasonable answer, and the agent never collects enough goal bonuses to learn otherwise.

Nothing in the algorithm changed. One number in the reward moved the behavior from reaching the goal in 96% of the test episodes to 5%. The reward is part of the problem specification. The agent maximizes the return that this reward defines, whether or not that matches the behavior you wanted. In practice, reward weights for navigation are tuned the way any other hyperparameter is, by training and watching what the robot does.

## From simulation to the robot

Everything above ran in a 2D simulation with perfect geometry. A policy trained this way meets several differences on a real TurtleBot3:

- The lidar is noisy, drops returns on dark or shiny surfaces, and sees chair legs and people that are not disks.
- The robot has inertia and motor limits, and its commands arrive with a delay, so $(v, \omega)$ is not applied exactly or at once.
- The goal's distance and bearing come from odometry or localization, which drift and jump.
- The room is new. The policy saw four fixed obstacles in a 4 m square.

The usual answers are a simulator closer to the robot, such as Gazebo with the TurtleBot3 model, which is what turtlebot3_drlnav uses, and randomization during training: random obstacle layouts, noise on the ranges and the commands, and random delays, so that the real robot looks like one more variation. Niu et al. trained a DDPG-based collision avoidance agent in Gazebo, added human teleoperation data to the replay buffer to cut the training time, and ran the trained policy on a real robot without further tuning [@Niu2021-st]. The state you used here, with lidar ranges, the goal's distance and bearing, and the previous command, is the turtlebot3_drlnav state. Moving to the real robot therefore changes the environment the agent trains in, while the agent's inputs, outputs and learning algorithm stay the same.